<h1><center>Data Science Rag</center></h1>

In [51]:
# --- Standard library ---
import os                       # environment variables
import json                     # parse embedding strings returned by Supabase
from pathlib import Path        # walk the knowledge base folder
from itertools import groupby
# --- Config & validation ---
from dotenv import load_dotenv            # load API keys from .env
from pydantic import BaseModel, Field     # structured-output schemas

# --- Vector store & RAG pipeline ---
from supabase import create_client, Client                         # Supabase connection
from langchain_community.vectorstores import SupabaseVectorStore   # pgvector store
from langchain_openai import OpenAIEmbeddings, ChatOpenAI          # embeddings + chat model
from langchain_core.prompts import ChatPromptTemplate              # prompt templates
from langchain_community.document_loaders import PyMuPDFLoader     # PDF document loader
from langchain_text_splitters import RecursiveCharacterTextSplitter # text splitter for chunking documents
from langchain_core.documents import Document
import re
import unicodedata
import tiktoken
# --- Utilities ---
from tqdm import tqdm           # progress bars
from collections import Counter # count occurrences of items in a list
# --- Visualization (optional) ---
import numpy as np                    # embedding matrix
import pandas as pd # data manipulation and analysis
from sklearn.manifold import TSNE     # reduce 1536-D vectors to 2-D/3-D
import plotly.graph_objects as go     # interactive scatter plots


In [41]:
load_dotenv(override=True)

# --- LLM ---
MODEL = "gpt-6-luna"            # bare model name for ChatOpenAI

# --- Embeddings ---
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIMENSIONS = 1536            # must match vector(1536) in Supabase; under the 2,000-dim HNSW limit

# --- Supabase vector store ---
TABLE_NAME = "documents"               # pgvector table
MATCH_FUNCTION = "match_documents"     # SQL similarity-search function

# --- Knowledge base & chunking ---
KNOWLEDGE_BASE_PATH = Path("data-science-knowledge-base")
CHUNK_SIZE_CHARS = 2000                # ~500 tokens (≈4 chars/token in English)
CHUNK_OVERLAP_CHARS = 200              # ~10% overlap so ideas aren't cut mid-thought

# --- Sanity checks: fail fast with clear messages ---
for var in ("OPENAI_API_KEY", "SUPABASE_URL", "SUPABASE_KEY"):
    assert os.getenv(var), f"Missing {var} in .env"
assert KNOWLEDGE_BASE_PATH.exists(), f"Knowledge base not found: {KNOWLEDGE_BASE_PATH.resolve()}"

# --- Clients ---
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMENSIONS)
supabase: Client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])

In [42]:
HEADER_RE = re.compile(r"^(?P<library>[\w\-]+)\s+(?P<version>\d+\.\d+(?:\.\d+)?)\s+reference\s*(?P<section>.*)$")
SOURCE_RE = re.compile(r"Source:\s*(https?://\S+)")
PUA_MAP = str.maketrans({"\uee47": ": ", "\uee4e": "(", "\uee4f": ")","\uee55": "-", "\uee6c": "x"})
VERSION_RE = re.compile(r"\d+\.\d+(?:\.\d+)?")

def clean_page(doc):
    """ Normalize text, lift section /version into metadata, capture Source URLs."""
    text = unicodedata.normalize("NFKC", doc.page_content).translate(PUA_MAP)
    text = re.sub(r" {2,}", " ", text)
    lines = text.splitlines()
    nonblank = [k for k, l in enumerate(lines) if l.strip()]
    section = None

    if nonblank:
        i = nonblank[0]
        m = HEADER_RE.match(lines[i].strip())
        if m:
            doc.metadata["version"] = m["version"]
            section = m["section"].strip()
            lines.pop(i)
            if not section:
                j = next((k for k in range(i, len(lines)) if lines[k].strip()), None)
                if j is not None and lines[j].strip().isupper():
                    section = lines.pop(j).strip()
        else:
            src = next((k for k in nonblank[:4] if lines[k].startswith("Source:")), None)
            if src is not None:
                section = " ".join(lines[k].strip() for k in range(i, src) if lines[k].strip())
            elif len(lines[i].strip()) < 60:
                section = lines[i].strip()
    if section and section.isupper():
        section = section.title()
    doc.metadata["section"] = section or None

    if "version" not in doc.metadata:
        v = VERSION_RE.search(doc.metadata.get("title", ""))
        doc.metadata["version"] = v.group() if v else None
    
    text = "\n".join(lines)
    urls = SOURCE_RE.findall(text)
    if urls:
        doc.metadata["source_url"] = urls[0]
    doc.page_content = text.strip()
    return doc

docs, skipped = [], 0
for pdf in sorted(KNOWLEDGE_BASE_PATH.glob("*.pdf")):
    for d in PyMuPDFLoader(str(pdf)).load():
        if d.page_content.lstrip().startswith("Contents"): # table-of-contents pages
            skipped += 1
            continue
        d.metadata["library"] = pdf.stem.replace("_reference", "")
        d.metadata["page"] = d.metadata.get("page", 0) + 1
        docs.append(clean_page(d))

print(f"{len(docs)} pages loaded, {skipped} contents pages skipped")




1040 pages loaded, 6 contents pages skipped


In [43]:
print(pd.DataFrame([{"library": d.metadata["library"], 
                    "chars": len(d.page_content)} for d in docs]).groupby("library").agg(pages=("chars", "size"), 
                    chars=("chars", "sum")), "\n")
glued = Counter(w for d in docs for w in re.findall(r"\b[A-Za-z]{25,}\b", d.page_content))
print(f"Glued words: {sum(glued.values()):,} (target ≈ 100 )")
print(glued.most_common(10), "\n")

print(f"Pages with section: {sum(bool(d.metadata.get('section')) for d in docs)}/{len(docs)}")
print(f"Pages with source_url: {sum(bool(d.metadata.get('source_url')) for d in docs)} (target ≈ 74)")
print(Counter(d.metadata.get("section") for d in docs).most_common(15))
print(docs[50].metadata)
print(docs[50].page_content[:600])


              pages   chars
library                    
matplotlib      129  264483
numpy           150  290448
pandas          219  434074
plotly          157  308187
scikit-learn    233  571257
scipy           152  334878 

Glued words: 104 (target ≈ 100 )
[('HistGradientBoostingClassifier', 33), ('GradientBoostingRegressor', 21), ('GradientBoostingClassifier', 14), ('HistGradientBoostingRegressor', 10), ('TransformedTargetRegressor', 9), ('InconsistentVersionWarning', 4), ('LinearDiscriminantAnalysis', 3), ('FillBetweenPolyCollection', 2), ('ParquetCompressionOptions', 1), ('DatetimeScalarOrArrayConvertible', 1)] 

Pages with section: 1040/1040
Pages with source_url: 74 (target ≈ 74)
[('Api Quick Reference', 289), ('Metrics And Scoring', 43), ('Time Series / Date Functionality', 36), ('Optimization (Scipy.Optimize)', 30), ('Numpy: The Absolute Basics', 29), ('Indexing And Selecting Data', 25), ('Linear Models', 25), ('Ensembles: Gradient Boosting, Random Forests, Bagging, Voting, St

In [44]:
# 1) How wide spread are Private Use Area characters, and what surrounds them?
pua = Counter(ch for d in docs for ch in d.page_content if 0xE000 <= ord(ch)<= 0xF8FF)
print(f"PUA chars in body text: {sum(pua.values()):,}")
for ch, n in pua.most_common(20):
    ex = next(d.page_content for d in docs if ch in d.page_content)
    k = ex.index(ch)
    print(f"U+{ord(ch):04X} x{n:>5}: {ex[max(0, k-35):k+35]!r}")

# 2) Which pages have no section?
none_pages = [d for d in docs if not d.metadata.get("section")]
print("\nNo-section pages by library:", Counter(d.metadata["library"] for d in none_pages))
for d in none_pages[:6]:
    print(d.metadata["library"], "p.", d.metadata["page"], repr(d.page_content[:120]))

PUA chars in body text: 0

No-section pages by library: Counter()


In [54]:
# --- Trim metadata to what retrieval and citations need ---
KEEP = ("library",
        "version",
        "section",
        "page",
        "source",
        "source_url")
for d in docs:
    d.metadata = {k: d.metadata.get(k) for k in KEEP}

# --- Split pages into overlapping chunks ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_CHARS,
    chunk_overlap=CHUNK_OVERLAP_CHARS,
    separators=["\n\n", "\n", " ", ""], # paragraph -> line -> word -> char
)
chunks = splitter.split_documents(docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(f"{len(docs)} pages-> {len(chunks)} chunks")
page_chunks = chunks

1040 pages-> 1652 chunks


In [55]:
enc = tiktoken.get_encoding("cl100k_base")
chunk_df = pd.DataFrame({
    "library": [c.metadata["library"] for c in chunks],
    "chars": [len(c.page_content) for c in chunks],
    "tokens": [len(enc.encode(c.page_content)) for c in chunks],
})

print(chunk_df.groupby("library").agg(chunks=("chars", "size"), mean_chars=("chars", "mean")).round(0), "\n")
print(chunk_df[["chars", "tokens"]].describe().round(0), "\n")

tiny = chunk_df[chunk_df.chars < 200]
print(f"Tiny chunks (<200 chars): {len(tiny)}")

PRICE_PER_1M = 0.13 # text-embedding-3-large, USD; confirm current OpenAI pricing
total_tokens = chunk_df.tokens.sum()
print(f"Total tokens: {total_tokens:,} -> est.embedding cost ${total_tokens / 1e6 * PRICE_PER_1M:.2f}")

go.Figure(go.Histogram(x=chunk_df.chars, nbinsx=40)).update_layout(
    title="Chunk length distribution", 
    xaxis_title="characters",
    yaxis_title="count"
).show()

              chunks  mean_chars
library                         
matplotlib       204      1359.0
numpy            220      1374.0
pandas           320      1409.0
plotly           223      1431.0
scikit-learn     434      1394.0
scipy            251      1401.0 

        chars  tokens
count  1652.0  1652.0
mean   1396.0   367.0
std     627.0   175.0
min      34.0    15.0
25%     793.0   199.0
50%    1686.0   428.0
75%    1956.0   496.0
max    2000.0  1101.0 

Tiny chunks (<200 chars): 14
Total tokens: 606,823 -> est.embedding cost $0.08


In [56]:
# Join consecutive chunks with the same library + section, so chunks flow across page breaks
sections = []
for _, group in groupby(docs, key=lambda d: (d.metadata["library"], d.metadata["section"])):
    group = list(group)
    meta = dict(group[0].metadata) # page = first page of the run
    meta["page_end"] = group[-1].metadata["page"]
    sections.append(Document(page_content="\n".join(g.page_content for g in group), metadata=meta))

print(f"{len(docs)} pages -> {len(sections)} section runs")
chunks = splitter.split_documents(sections)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i
print(f"{len(sections)} sections runs -> {len(chunks)} chunks")

1040 pages -> 170 section runs
170 sections runs -> 1294 chunks


In [57]:
enc = tiktoken.get_encoding("cl100k_base")
chunk_df = pd.DataFrame({
    "library": [c.metadata["library"] for c in chunks],
    "chars": [len(c.page_content) for c in chunks],
    "tokens": [len(enc.encode(c.page_content)) for c in chunks],
})

print(chunk_df.groupby("library").agg(chunks=("chars", "size"), mean_chars=("chars", "mean")).round(0), "\n")
print(chunk_df[["chars", "tokens"]].describe().round(0), "\n")

tiny = chunk_df[chunk_df.chars < 200]
print(f"Tiny chunks (<200 chars): {len(tiny)}")

PRICE_PER_1M = 0.13 # text-embedding-3-large, USD; confirm current OpenAI pricing
total_tokens = chunk_df.tokens.sum()
print(f"Total tokens: {total_tokens:,} -> est.embedding cost ${total_tokens / 1e6 * PRICE_PER_1M:.2f}")

go.Figure(go.Histogram(x=chunk_df.chars, nbinsx=40)).update_layout(
    title="Chunk length distribution", 
    xaxis_title="characters",
    yaxis_title="count"
).show()

              chunks  mean_chars
library                         
matplotlib       161      1778.0
numpy            171      1846.0
pandas           249      1896.0
plotly           183      1817.0
scikit-learn     334      1864.0
scipy            196      1855.0 

        chars  tokens
count  1294.0  1294.0
mean   1849.0   489.0
std     362.0   128.0
min     199.0    52.0
25%    1931.0   445.0
50%    1964.0   492.0
75%    1984.0   549.0
max    2000.0  1087.0 

Tiny chunks (<200 chars): 1
Total tokens: 632,452 -> est.embedding cost $0.08


In [46]:
class Result(BaseModel):
    """ Result is the notebooks universal 'retrieval chunk' container
    page_content: the text that gets embedded, searched and parsed into prompts.
    metadata: where the text came from (source file path + document type)
    """
    page_content: str
    metadata:dict

class Chunk(BaseModel):
    headline: str = Field(description='A brief heading for this chunk. typically a few words, that is most likely to be surfaced in a query')
    summary: str = Field(description='A few sentences summarizing the content of this chunk to answer common queries')
    original_text: str = Field(description='The original text content of this chunk before any processing or summarization')

    def as_result(self, document):
        metadata = {
            'source': document['source'],
            'type': document['type']}